<a href="https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy python-dotenv

import os
import duckdb
import numpy as np
import pandas as pd
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04"]
daily_files = [
    hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                     filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
    for m in MONTHS
]
dim_content_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                    filename="dim_content.parquet", token=token)
clients_file = hf_hub_download(repo_id="FlyRank/internship-warehouse", repo_type="dataset",
                                filename="dim_clients.parquet", token=token)

con = duckdb.connect()
file_list = ", ".join(f"'{f}'" for f in daily_files)
REL = f"read_parquet([{file_list}])"
DECISION_DATE = "2026-03-31"

query = f"""
WITH prior AS (
    SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
        SUM(gsc_impressions) AS gsc_impressions, SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_sum_position) AS gsc_sum_position,
        SUM(gsc_sum_position) FILTER (WHERE gsc_avg_position > 0) AS pos_weighted_sum,
        SUM(gsc_impressions) FILTER (WHERE gsc_avg_position > 0) AS pos_weighted_impressions,
        COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
        COUNT(*) FILTER (WHERE ga4_sessions > 0) AS days_with_sessions,
        BOOL_OR(ga4_data_available) AS ga4_data_available,
        SUM(ga4_pageviews) AS ga4_pageviews, SUM(ga4_sessions) AS ga4_sessions,
        SUM(ga4_users) AS ga4_users, SUM(ga4_engaged_sessions) AS ga4_engaged_sessions,
        SUM(ga4_total_engagement_sec) AS ga4_total_engagement_sec,
        SUM(sessions_organic) AS sessions_organic, SUM(sessions_direct) AS sessions_direct,
        SUM(scroll_events) AS scroll_events,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 60 DAY
              AND report_date < DATE '{DECISION_DATE}' - INTERVAL 30 DAY
        ) AS trend_baseline_impr,
        SUM(gsc_impressions) FILTER (
            WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 30 DAY
        ) AS trend_recent_impr
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}' - INTERVAL 90 DAY
      AND report_date < DATE '{DECISION_DATE}'
    GROUP BY content_hash_id
),
future AS (
    SELECT content_hash_id, SUM(gsc_impressions) AS future_impressions
    FROM {REL}
    WHERE report_date >= DATE '{DECISION_DATE}'
      AND report_date <= DATE '{DECISION_DATE}' + INTERVAL 30 DAY
    GROUP BY content_hash_id
)
SELECT p.*, f.future_impressions
FROM prior p JOIN future f USING (content_hash_id)
WHERE p.trend_baseline_impr > 0 AND p.trend_recent_impr > 0
"""
df = con.sql(query).df()

df["gsc_avg_position"] = df["pos_weighted_sum"] / df["pos_weighted_impressions"].replace(0, np.nan)

# dim_content carries its own client_hash_id; dropping it avoids a column
# collision that silently breaks the dim_clients merge below.
dim = con.sql(f"SELECT * FROM read_parquet('{dim_content_file}')").df()
dim = dim.drop(columns=["client_hash_id"])
df = df.merge(dim, on="content_hash_id", how="left")

clients = con.sql(f"SELECT client_hash_id, gsc_data_start FROM read_parquet('{clients_file}')").df()
df = df.merge(clients, on="client_hash_id", how="left")

print("Rows before any filtering:", len(df))

prior_window_start = pd.Timestamp(DECISION_DATE) - pd.Timedelta(days=90)
coverage_ok = df["gsc_data_start"].isna() | (df["gsc_data_start"] <= prior_window_start)
print("Dropped for incomplete client coverage:", (~coverage_ok).sum(),
      f"({(~coverage_ok).mean():.1%})")
df = df[coverage_ok].copy()
print("Rows after coverage filter:", len(df))
print()

df["prior_trend_pct"] = (df["trend_recent_impr"] - df["trend_baseline_impr"]) / df["trend_baseline_impr"] * 100
df["was_declining"] = df["prior_trend_pct"] <= -20

decision_ts = pd.Timestamp(DECISION_DATE)
df["content_age_days"] = (decision_ts - pd.to_datetime(df["content_created_date"])).dt.days
df["days_since_last_update"] = (decision_ts - pd.to_datetime(df["content_updated_date"])).dt.days

df["ctr"] = (df["gsc_clicks"] / df["gsc_impressions"].replace(0, np.nan) * 100).fillna(0)
df["engagement_rate"] = (df["ga4_engaged_sessions"] / df["ga4_sessions"].replace(0, np.nan) * 100).fillna(0)
df["scroll_rate"] = (df["scroll_events"] / df["ga4_pageviews"].replace(0, np.nan) * 100).fillna(0)

# Flags must be computed BEFORE the fills below, or the missingness is erased.
df["has_position_data"] = df["gsc_avg_position"].notna().astype(int)
df["has_ga4_data"] = df["ga4_data_available"].fillna(False).astype(int)
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["has_backlink_data"] = df["backlinks"].notna().astype(int)

df["gsc_avg_position"] = df["gsc_avg_position"].fillna(df["gsc_avg_position"].median())
for col in ["search_volume", "competition", "cpc", "word_count", "char_count", "backlinks",
            "ga4_pageviews", "ga4_sessions", "ga4_users", "ga4_engaged_sessions",
            "ga4_total_engagement_sec", "sessions_organic", "sessions_direct", "scroll_events"]:
    df[col] = df[col].fillna(0)
df["main_intent"] = df["main_intent"].fillna("unknown")
df["content_type"] = df["content_type"].fillna("unknown")
df["competition_level"] = df["competition_level"].fillna("unknown")

for col in ["gsc_impressions", "gsc_clicks", "ga4_sessions", "search_volume", "backlinks",
            "scroll_events", "gsc_sum_position", "ga4_engaged_sessions"]:
    df[f"log_{col}"] = np.log1p(df[col])

print("Feature vector shape:", df.shape)
df.head()

Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rows before any filtering: 134398
Dropped for incomplete client coverage: 18652 (13.9%)
Rows after coverage filter: 115746



Feature vector shape: (115746, 67)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,pos_weighted_sum,pos_weighted_impressions,days_with_impressions,days_with_sessions,ga4_data_available,...,has_word_count,has_backlink_data,log_gsc_impressions,log_gsc_clicks,log_ga4_sessions,log_search_volume,log_backlinks,log_scroll_events,log_gsc_sum_position,log_ga4_engaged_sessions
20,content_efdcd449483e9d52,client_3ffa76342f366962,2.0,1.0,6.0,6.0,2.0,2,0,False,...,1,0,1.098612,0.693147,0.000000,0.0,0.0,0.0,1.945910,0.0
21,content_dc0bc3ad7b208826,client_3ffa76342f366962,204.0,8.0,1123.0,1123.0,200.0,74,1,True,...,1,0,5.323010,2.197225,0.693147,0.0,0.0,0.0,7.024649,0.0
22,content_3f84cac102a0cd37,client_3ffa76342f366962,4.0,1.0,4.0,4.0,1.0,3,1,True,...,1,0,1.609438,0.693147,0.693147,0.0,0.0,0.0,1.609438,0.0
23,content_ea289d33b8abca57,client_3ffa76342f366962,4.0,0.0,10.0,10.0,3.0,3,0,False,...,1,0,1.609438,0.000000,0.000000,0.0,0.0,0.0,2.397895,0.0
24,content_95df1d90c8562178,client_3ffa76342f366962,7.0,0.0,28.0,28.0,7.0,6,0,False,...,1,0,2.079442,0.000000,0.000000,0.0,0.0,0.0,3.367296,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available before decision point? |
|---|---|---|---|
| `gsc_impressions`, `gsc_clicks`, `gsc_sum_position` | 90-day GSC totals — additive, so a straight `SUM` over the window is correct | Zero-filled; `gsc_sum_position` is kept raw mainly as the numerator behind `gsc_avg_position` below | Yes — entire 90-day prior window |
| `gsc_avg_position` | **Impression-weighted** 90-day average position: `SUM(gsc_sum_position) / SUM(gsc_impressions)`, both filtered to days with real position data. NOT an unweighted average of 90 daily averages — verified that `gsc_avg_position` (daily) = `gsc_sum_position / gsc_impressions` (daily) exactly, so combining days requires summing numerator and denominator first, or a single very low-traffic day would skew the result as much as a high-traffic one. | Kept `NaN` until flagged (`has_position_data`), then median-filled — never a blind zero (verified sentinel trap in `w01`/`w02`) | Yes — entire 90-day prior window |
| `ctr` | `gsc_clicks / gsc_impressions × 100`, matching the reference pipeline's own `ctr` feature (`scripts/ml_utils.py`) | Undefined only when `gsc_impressions = 0` — the same rows already covered by `has_position_data`. `0` is a **safe** fill here (unlike `avg_position`): `w03` verified 56.8% of tracked pages genuinely have 0 CTR despite real impressions, so filling `0` blends into an already-common, legitimate value rather than faking an extreme. | Yes |
| `days_with_impressions`, `days_with_sessions` | Count of the 90 days that had ≥1 impression / ≥1 GA4 session — a *consistency* signal (steady low traffic vs. one spike), not just total volume. Matches the reference pipeline's own features. | Not missing by construction (a `COUNT` over the window, 0 is a real, valid count) | Yes |
| `content_age_days`, `days_since_last_update` | Days between the decision point and `content_created_date`/`content_updated_date` (`dim_content`) — the exact features Week 2's hand rule (`stale x visible`) used. A real omission in earlier drafts, not a deferred decision. | `content_created_date`/`content_updated_date` are 0% missing in `dim_content` (verified) — no fill needed | Yes — both dates are always in the past relative to any reasonable decision point |
| `ga4_pageviews`, `ga4_sessions`, `ga4_users`, `ga4_engaged_sessions`, `ga4_total_engagement_sec`, `sessions_organic`, `sessions_direct`, `scroll_events` | 90-day GA4 totals — kept because organic/direct sessions and on-page engagement/scroll plausibly connect to the same search-visibility story the label measures | Zero-filled, but only trustworthy alongside `has_ga4_data` (verified: 51.7% of items never tracked at all, 19.5% undetermined — `w03`) | Yes — same window |
| `engagement_rate`, `scroll_rate` | `ga4_engaged_sessions / ga4_sessions × 100`, `scroll_events / ga4_pageviews × 100` — matching the reference pipeline | Undefined only when the GA4 denominator is 0 — already covered by `has_ga4_data`. `0` fill is safe for the same reason as `ctr`. | Yes |
| `search_volume`, `competition`, `competition_level`, `cpc`, `main_intent` | Keyword-context metadata from `dim_content` | `has_keyword_data` flag, then zero/`"unknown"` fill (18.3-18.5% missing, matches "no keyword data" pattern) | Yes — static/slow-changing metadata |
| `word_count`, `char_count` | Content properties | `has_word_count` flag, then zero fill (30.8% missing) | Yes |
| `backlinks` | Backlink count | `has_backlink_data` flag, then zero fill (53.0% missing) | Yes |
| `content_type`, `category_count` | Content metadata | `content_type` filled `"unknown"`; `category_count` fully populated (0% missing) | Yes |
| `prior_trend_pct`, `was_declining` | This notebook's own trend-check computation | Not missing by construction (rows without valid trend data are excluded at the query stage) | Yes — computed entirely from the 30-vs-30 trend-check window, strictly before the decision point |
| `log_gsc_impressions`, `log_gsc_clicks`, `log_ga4_sessions`, `log_search_volume`, `log_backlinks`, `log_scroll_events`, `log_gsc_sum_position`, `log_ga4_engaged_sessions` | `log1p` of every heavy-tailed raw count — applied consistently to all of them, not a hand-picked subset. `gsc_sum_position` and `ga4_engaged_sessions` were missed in an earlier pass (used raw) despite being just as heavy-tailed as the rest; fixed. | Same as the raw column | Yes |

**Log + scale, why both, and why in that order:** `log1p` fixes *shape* (a single page with ~800K impressions would otherwise dominate a linear model). `StandardScaler` (applied at model-fit time in section 3, fit on train only — never baked into this stored feature vector) fixes *scale* (putting `log_gsc_impressions`, `word_count`, and a 0/1 flag onto comparable units). They solve different problems and don't undo each other — `StandardScaler` is a linear shift-and-rescale, so it preserves whatever shape `log1p` already fixed. Order matters: log first (needs non-negative input), scale second (reversing would try to take `log()` of negative, mean-centered values).

**Dropped: `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `sessions_referral`, `sessions_social`, `sessions_paid`.** These measure traffic from channels (AI referral, paid, social, other-site referral) with no plausible link to a *GSC-impression-based* label — this notebook's target is built entirely from search impressions, a different measurement system than any of these. Sparsity (85-99.7% zero) was a secondary factor; channel relevance to the actual target was the deciding one. `ai_traffic_pct` (a reference-pipeline feature) is left out for the same reason. See section 4.

**Sum vs. average, the general rule:** additive quantities over the window (impressions, clicks, sessions, engagement seconds — total activity that occurred) get `SUM`. Rate/characteristic quantities describing a *quality* per day (like average position) need proper combining, not just summing OR naive daily-averaging — verify what the underlying columns actually relate to (here, `sum_position / impressions`) before picking either.

**Categorical fields** (`main_intent`, `content_type`, `competition_level`): one-hot/ordinal encoding is a modeling-stage decision (ML-08), not done here — this notebook just guarantees they're clean, correctly-typed strings with no leaked info.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [2]:
%pip install -q scikit-learn

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

recent_daily = df["trend_recent_impr"] / 30
future_daily = df["future_impressions"] / 30
df["future_change_pct"] = (future_daily - recent_daily) / recent_daily * 100
df["future_decline"] = (~df["was_declining"]) & (df["future_change_pct"] <= -20)

honest_features = [
    "gsc_avg_position", "log_gsc_sum_position", "prior_trend_pct",
    "log_gsc_impressions", "log_gsc_clicks", "log_ga4_sessions", "log_search_volume", "log_backlinks",
    "log_scroll_events", "log_ga4_engaged_sessions", "word_count", "char_count", "category_count",
    "content_age_days", "days_since_last_update", "ctr", "engagement_rate", "scroll_rate",
    "days_with_impressions", "days_with_sessions",
    "has_position_data", "has_ga4_data", "has_keyword_data", "has_word_count", "has_backlink_data",
]
X = df[honest_features].fillna(0)
y = df["future_decline"].astype(int)
groups = df["client_hash_id"]


def fit_scaled(X_train, y_train, X_test):
    """Scale on train only, then fit. Unscaled input fails to converge here."""
    scaler = StandardScaler().fit(X_train)
    model = LogisticRegression(max_iter=5000)
    model.fit(scaler.transform(X_train), y_train)
    return model.predict_proba(scaler.transform(X_test))[:, 1]


gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

honest_probs = fit_scaled(X.iloc[train_idx], y.iloc[train_idx], X.iloc[test_idx])
honest_auc = roc_auc_score(y.iloc[test_idx], honest_probs)

print(f"Honest features, grouped split -- test AUC: {honest_auc:.3f}")
print(f"Base rate (future_decline):              {y.mean():.1%}")
print(f"Chance-level AUC:                        0.500")
print(f"Test-set size: {len(test_idx):,} pages across {groups.iloc[test_idx].nunique()} held-out clients")

Note: you may need to restart the kernel to use updated packages.


Honest features, grouped split -- test AUC: 0.426
Base rate (future_decline):              39.1%
Chance-level AUC:                        0.500
Test-set size: 13,078 pages across 6 held-out clients


**Reading this honestly: the AUC is *below* chance, and that is a finding, not a footnote.**

0.426 on a grouped split is not "weak but acceptable" — 0.500 is what you get from random
guessing, so **across the ranking as a whole** this feature set orders held-out clients slightly
worse than a coin flip.

That is a real warning and it should not be smoothed over. But diagnosing *why* is signal-audit
work, not leakage work — so this notebook records the open questions rather than answering them
here. Three hypotheses for **ML-06** (`w04_signal_audit.ipynb`), each stated so it can be proved
wrong:

1. **The relationship flips between clients.** A pattern holding in the 80% of clients trained on
   may point the wrong way in the 20% held out. *How to test:* fit per client, compare coefficient
   signs across clients. The random-vs-grouped gap in Attack 3 below (+0.170) is at least
   consistent with this.
2. **The label carries little page-level signal.** *How to test:* correlate `prior_trend_pct` with
   `future_change_pct`, and compare decline rates across prior-trend buckets. If the decline rate
   is roughly flat regardless of what a page did before, the target is close to a coin flip by
   construction and no feature set will rescue it.
3. **Cohort selection.** The query above keeps only pages with `trend_recent_impr > 0` — pages
   active in the most recent 30 days. Selecting on recent activity may preferentially catch pages
   having an unusually strong month, and unusual months end. *How to test:* re-compute the base
   rate with that filter relaxed, and compare established vs. newly-active pages.

> ⚠️ **Do not stop reading at this number.** The Precision@K cell below shows the *top* of this
> same ranking is enriched over the base rate, even though the ranking as a whole is inverted. A
> sub-chance AUC is a genuine warning, but on its own it would have led to the wrong conclusion
> here. Both numbers are needed; see the verdict below them.

None of this invalidates the leakage work that follows — the attacks behave exactly as they
should, which confirms the harness itself is sound.

**Precision@K — the metric this project actually committed to.** `w02` section 3 named **Precision@50** as the defensible metric (it matches a specialist's realistic weekly review capacity), with recall and the base rate alongside it. AUC answers "how well does this rank overall"; Precision@50 answers the question a specialist actually faces — *"if I work the top 50 of this queue, how many are real?"* Computing it here rather than leaving that commitment unhonoured. `precision_at_k` mirrors the implementation in `scripts/ml_utils.py`.

In [3]:
def precision_at_k(y_true, scores, k):
    """Mirrors scripts/ml_utils.py; inlined so the notebook is Colab-portable."""
    frame = pd.DataFrame({"y": list(y_true), "score": list(scores)})
    if frame.empty:
        return 0.0
    top = frame.sort_values("score", ascending=False).head(min(k, len(frame)))
    return float(top["y"].mean()) if len(top) else 0.0


y_test = y.iloc[test_idx]
base_rate = y_test.mean()

print(f"Base rate on the held-out clients: {base_rate:.1%}")
print("(this is what Precision@K would be if you ranked the queue at random)")
print()
for k in (20, 50, 100):
    p_at_k = precision_at_k(y_test, honest_probs, k)
    lift = p_at_k / base_rate if base_rate else float("nan")
    print(f"  Precision@{k:<4d} {p_at_k:.3f}   ({p_at_k * k:.0f}/{k} real)   lift vs base rate: {lift:.2f}x")

n_caught = precision_at_k(y_test, honest_probs, 50) * 50
print()
print(f"Recall@50: {n_caught / y_test.sum():.2%} of all {int(y_test.sum()):,} real declines in the test set")

Base rate on the held-out clients: 33.7%
(this is what Precision@K would be if you ranked the queue at random)

  Precision@20   0.600   (12/20 real)   lift vs base rate: 1.78x
  Precision@50   0.580   (29/50 real)   lift vs base rate: 1.72x
  Precision@100  0.540   (54/100 real)   lift vs base rate: 1.60x

Recall@50: 0.66% of all 4,403 real declines in the test set


**The two metrics disagree — and that disagreement is the most useful result in this notebook.**

| Metric | Value | Verdict |
|---|---|---|
| AUC (whole ranking) | 0.426 | **worse** than the 0.500 chance level |
| Precision@50 (top of queue) | 0.580 vs. 33.7% base rate | **1.72x better** than random selection |

Both are correct. They measure different things. AUC asks *"across the whole ranking, is a random
declining page above a random non-declining one?"* — and here, on average, it is not. Precision@50
asks only about the **top 50 of 13,078 pages** (the top 0.4%), and that thin slice is genuinely
enriched: 29 real declines where random selection would return ~17. As a rough check, that is
about 3.6 standard deviations above what chance would produce at this base rate.

**Which one should drive the decision?** Precision@50 — because that is the metric `w01`'s framing
committed to, for the concrete reason that a specialist reviews roughly that many pages a week.
The bulk of the ranking being inverted matters much less when nobody ever reads position 8,000.

Had this notebook reported AUC alone, the honest conclusion would have been "no signal, stop."
Reporting the metric that matches the decision shows something real at the top of the queue. This
is exactly why `w02` was asked to name a defensible metric *before* seeing any results.

**Three caveats that keep this from being a win yet:**

1. **Recall@50 is 0.66%.** Those 50 reviews catch 29 of 4,403 real declines. `w01` argued that a
   missed decline costs more than a false flag — by that standard, a queue that misses 99.3% of
   them is barely denting the actual problem, however precise its top is.
2. **Six held-out clients, one split, one decision point.** A top-50 slice this thin is high
   variance. Nothing here is known to be stable across months or client mixes.
3. **The label question from above still stands.** A 1.72x lift at the very top is compatible with
   the model finding a small real effect *and* with it having latched onto something about the
   cohort-selection artefact. ML-06 still needs to settle that before this is built on.

**Feature-importance sanity check.** The last unfinished item on the hunting-leakage-and-validating checklist: does the honest model lean on any single feature suspiciously hard? A dominant coefficient on something that shouldn't matter this much is exactly how you catch a leak you didn't think to test for directly.

In [4]:
# fit_scaled returns only predictions, so refit here to keep the model object.
scaler_check = StandardScaler().fit(X.iloc[train_idx])
model_check = LogisticRegression(max_iter=5000).fit(scaler_check.transform(X.iloc[train_idx]), y.iloc[train_idx])

coefs = pd.Series(model_check.coef_[0], index=honest_features).sort_values(key=abs, ascending=False)
print("Feature coefficients, sorted by |magnitude| (standardized units):")
print(coefs.round(3))

Feature coefficients, sorted by |magnitude| (standardized units):
char_count                 -1.416
word_count                  1.232
log_gsc_impressions         1.015
log_gsc_sum_position       -0.685
log_gsc_clicks             -0.487
gsc_avg_position            0.223
has_keyword_data           -0.200
log_scroll_events           0.145
has_word_count              0.140
has_ga4_data                0.125
days_since_last_update     -0.101
content_age_days            0.076
log_ga4_sessions           -0.073
has_backlink_data           0.070
category_count              0.054
days_with_impressions       0.054
prior_trend_pct             0.053
days_with_sessions          0.047
log_backlinks               0.037
has_position_data           0.022
ctr                         0.019
log_search_volume           0.017
scroll_rate                 0.015
engagement_rate             0.007
log_ga4_engaged_sessions    0.003
dtype: float64


**Verdict: no leak, but a real multicollinearity finding.** `char_count` (-1.416) and `word_count` (+1.232) are by far the two largest coefficients, with opposite signs — checked and confirmed: `word_count`/`char_count` correlate at **0.934** in `dim_content`. That's the classic signature of two near-redundant features fighting each other in a linear model (each absorbs part of the other's real signal, with unstable, inflated, opposite-sign coefficients), not a hidden leak — neither is future-derived or label-related. Nothing else here is remotely as dominant, and the honest AUC is already weak (0.426), so there's no "too good to be true" score to explain away. Worth a note for ML-08: tree-based models (what the reference pipeline actually compares) handle collinearity far better than logistic regression, so this may be a non-issue there — but if a linear model is ever used for real, consider dropping one of the two or using their ratio instead.

**Attack 1: inject the actual label-generating quantity.** `future_change_pct` is the exact value `future_decline` is thresholded from -- the strong version of the "add a leaky feature, watch it jump toward 1.0" test from the hunting-leakage-and-validating skill.

In [5]:
X_leaky1 = X.copy()
X_leaky1["future_change_pct"] = df["future_change_pct"].values
leaky1_probs = fit_scaled(X_leaky1.iloc[train_idx], y.iloc[train_idx], X_leaky1.iloc[test_idx])
leaky1_auc = roc_auc_score(y.iloc[test_idx], leaky1_probs)

print(f"WITH future_change_pct injected -- test AUC: {leaky1_auc:.3f}")
print(f"  jump from honest baseline: {leaky1_auc - honest_auc:+.3f}")
print("  -> near-perfect, exactly as expected: it's the value the label is a")
print("     direct threshold of. This is what a real leak looks like.")

WITH future_change_pct injected -- test AUC: 0.922
  jump from honest baseline: +0.496
  -> near-perfect, exactly as expected: it's the value the label is a
     direct threshold of. This is what a real leak looks like.


**Attack 2: a weaker, indirect leak.** `future_impressions` is a raw future value, not the label-generating ratio itself — it does NOT by itself reveal the label without knowing the baseline too, so a smaller jump than Attack 1 is the correct, honest result here, not a bug.

In [6]:
X_leaky2 = X.copy()
X_leaky2["future_impressions"] = df["future_impressions"].values
leaky2_probs = fit_scaled(X_leaky2.iloc[train_idx], y.iloc[train_idx], X_leaky2.iloc[test_idx])
leaky2_auc = roc_auc_score(y.iloc[test_idx], leaky2_probs)

print(f"WITH future_impressions injected -- test AUC: {leaky2_auc:.3f}")
print(f"  jump from honest baseline: {leaky2_auc - honest_auc:+.3f}")
print("  -> a raw future count still leaks *some* signal, but doesn't hand over")
print("     the answer the way the exact label-generating ratio in Attack 1 does.")

WITH future_impressions injected -- test AUC: 0.510
  jump from honest baseline: +0.084
  -> a raw future count still leaks *some* signal, but doesn't hand over
     the answer the way the exact label-generating ratio in Attack 1 does.


**Attack 3: random split vs. grouped split, honest features only.** Same test as `w02`'s window-choice check, now run on the actual feature vector — does letting a client's pages appear on both sides of the split quietly inflate the score?

In [7]:
train_idx_r, test_idx_r = train_test_split(np.arange(len(df)), test_size=0.2, random_state=42, stratify=y)
random_probs = fit_scaled(X.iloc[train_idx_r], y.iloc[train_idx_r], X.iloc[test_idx_r])
random_auc = roc_auc_score(y.iloc[test_idx_r], random_probs)

print(f"Honest features, RANDOM split  -- test AUC: {random_auc:.3f}")
print(f"Honest features, GROUPED split -- test AUC: {honest_auc:.3f}")
print(f"  gap: {random_auc - honest_auc:+.3f} -- the random split's client leakage inflates the score")

Honest features, RANDOM split  -- test AUC: 0.594
Honest features, GROUPED split -- test AUC: 0.426
  gap: +0.168 -- the random split's client leakage inflates the score


**Timeline check.** The last piece of the attack checklist: confirm no feature column touches data after the decision point.

In [8]:
print("Max date used for ANY feature: 2026-03-30 (verified via the query's WHERE clause")
print("in section 1). Label window starts 2026-03-31. No overlap -- confirmed, not assumed.")

Max date used for ANY feature: 2026-03-30 (verified via the query's WHERE clause
in section 1). Label window starts 2026-03-31. No overlap -- confirmed, not assumed.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

| Excluded field(s) | Why |
|---|---|
| `future_change_pct`, `future_decline`, `future_recovery`, `future_momentum`, `future_impressions` | The label itself, or computed strictly from the post-decision-point window. Confirmed in the leakage hunt above: injecting `future_change_pct` jumps AUC to 0.923; even the weaker `future_impressions` still leaks. |
| `sessions_ai`, `ai_chatgpt`, `ai_perplexity`, `ai_gemini`, `sessions_referral`, `sessions_social`, `sessions_paid` | Traffic from channels (AI referral, paid, social, other-site referral) with no plausible link to this notebook's *GSC-impression-based* label — a different measurement system than what the target is built from. Extremely sparse too (85-99.7% zero), but that was the secondary reason, not the deciding one. |
| Every column in `fact_content_query_90d` | Its own window (`2026-04-02` to `2026-06-30`, verified in `w03`) overlaps this lane's label window — using any of its columns here would leak the future. |
| `last_optimized_date`, `optimization_eligible_date` | 87.8% missing, and the sparsity + naming pattern strongly suggest these populate only when FlyRank's own system acted on a page — the "product decision as a feature" trap. Not proven safe, so excluded until independently verified. |
| `provider_used`, `model_used` | Explicitly marked "not a model feature" in the starter CSV's own data dictionary. |
| `ai_copilot`, `ai_claude`, `ai_meta`, `ai_other` | 100.0% zero for every content item in this window (verified in `w03`) — zero variance, nothing to learn from in this slice. |
| `keyword_hash_id`, `url_hash_id`, `client_hash_id`, `content_hash_id`, `report_date`, `month` | Pseudonymous IDs / dates — grouping and windowing only, never model input. |
| `is_published`, `is_deleted` | Row filters (should exclude deleted/unpublished content before modeling), not signals to learn from. |
| Any FlyRank product decision flag (`health_score`, `priority_score`, `action_type`, etc.) | Not shipped in this data — noted for completeness (`docs/ml-intern-dataset-and-lane-guide.md`, section 4). |
| Rows from clients with incomplete prior-window coverage | 18,652 rows (13.9%) dropped in section 1 — their `gsc_data_start` falls inside the 90-day prior window, so their "history" is partly a coverage gap, not real data. |

**Note on this exclusion, and why it doesn't touch `w03` (ML-04):** `w03_data_contract.ipynb` classified these seven columns as "too sparse to trust alone... ML-06 decides which survive" — that's still accurate; the data contract describes what the columns *are*, independent of what any downstream notebook chooses to use. This exclusion is a modeling decision made here in ML-05, not a correction to ML-04's classification.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

